# Notebook 08: Autoencoders - Learning to Compress and Reconstruct

---

## What This Notebook Covers

This notebook introduces **Autoencoders**, a type of neural network that learns to compress data into a smaller representation and then reconstruct it. We will learn:

1. **What is an Autoencoder?** - The concept and architecture
2. **Why use Autoencoders?** - Applications and use cases
3. **Fashion MNIST Dataset** - Loading and exploring the data
4. **CNN Classification (Warmup)** - Review of convolutional networks
5. **Building an Autoencoder** - Encoder and Decoder design
6. **Deconvolution (Upsampling)** - How to increase image size
7. **Training and Evaluation** - MSE loss for reconstruction

---

## What is an Autoencoder?

An **Autoencoder** is a neural network that learns to:
1. **Encode**: Compress input data into a smaller representation (called the **latent space** or **bottleneck**)
2. **Decode**: Reconstruct the original input from this compressed representation

```
Input Image          Encoder          Latent Space         Decoder         Reconstructed Image
   28×28      →    [Compress]    →      Small       →    [Expand]     →        28×28
   784 pixels      (CNN layers)      (e.g., 64 values)   (Deconv layers)      784 pixels
```

**The key insight:** The network is forced to learn the most important features of the data because it must squeeze everything through a small bottleneck!

---

## Why Use Autoencoders?

| Application | How It Works |
|-------------|-------------|
| **Dimensionality Reduction** | The latent space is a compressed representation |
| **Denoising** | Train on noisy inputs, reconstruct clean outputs |
| **Anomaly Detection** | Unusual inputs have high reconstruction error |
| **Feature Learning** | The encoder learns useful representations |
| **Generative Models** | Variational Autoencoders (VAEs) can generate new data |

---

## Google Colab Setup

Run the cells below **once** at the start of each Colab session. They mount Google Drive, set the working directory, install the standard packages, and clone the `miniai` library from the fast.ai Part 2 course repo so the `from miniai.X import *` lines work.

**GPU note:** Training is meaningful only on GPU. Go to **Runtime &rarr; Change runtime type &rarr; GPU (T4)** before running training cells.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive/Fast.AI_Colab')
print(os.getcwd())

/content/drive/MyDrive/Fast.AI_Colab


In [3]:
# Install required packages
!pip install -q fastcore fastai diffusers datasets torcheval accelerate

# Clone the fast.ai Part 2 course repo to get the miniai library
if not os.path.exists('course22p2'):
    !git clone https://github.com/fastai/course22p2.git

# Add miniai to the Python path
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'course22p2'))

# Verify miniai is accessible
try:
    import miniai
    print(f'miniai loaded successfully from: {miniai.__file__}')
except ImportError:
    print('ERROR: miniai not found. Check that course22p2 was cloned correctly.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 7.3 MB/s eta 0:00:00
miniai loaded successfully from: /content/drive/MyDrive/Fast.AI_Colab/course22p2/miniai/__init__.py


---

*The cells above are Colab-specific setup. The content below is the same as the source notebook (`08_autoencoder_explained.ipynb`), unchanged.*

---

---

## 🎬 Interactive Visualizations in This Notebook

This notebook now includes **six hands-on interactive visualizations** that you can click, drag, step through, and play. Each one targets a concept that is much easier to *see* than to read about:

| # | Visualization | What it makes clear |
|---|---------------|---------------------|
| 1 | **Autoencoder Pipeline Explorer** | The whole squeeze-then-rebuild story end to end, with the 784 → 256 → 784 value counts |
| 2 | **Convolution Stride Lab** | How `stride=2` halves the image and how filters create channels |
| 3 | **Deconv Upsample Lab** | The two-step `upsample → conv` recipe, plus the checkerboard problem |
| 4 | **"Same" Padding Playground** | Why `padding = ks//2` keeps the spatial size identical |
| 5 | **3D Architecture Hourglass** | The tensor volumes shrinking and expanding in 3D (drag to rotate) |
| 6 | **Latent Space Explorer** | Dragging through the compressed code and watching reconstructions morph |

Most have **clickable stages**, a **speed slider**, **Play/Step** controls, and a **code panel** that updates to show the exact line running at each stage. Just run the cell beneath each section. *(They are self-contained and work offline.)*

---

### 🔍 Visualize It: The Full Autoencoder Pipeline

Before any code, here is the big picture. Step through the five stages (or press **Play**) to watch an image get squeezed through the bottleneck and rebuilt. Click any stage on the rail to jump to it and see the code for that stage.

In [4]:
# ============================================================================
# INTERACTIVE VISUALIZATIONS -- setup (run this cell ONCE).
# Each "🎮 Interactive" cell below loads a standalone HTML file from the
# published copy on GitHub Pages, in an isolated <iframe> shown FULL WIDTH that
# auto-fits its content height. No inline HTML -- the visualizations are read
# from GitHub, so the cells stay short.
# ============================================================================
from IPython.display import HTML

def show_viz(path, height="600px"):
    """Embed an interactive visualization full width; it auto-fits its height.
    `path` may be a bare 'interactive_viz/<file>.html' (resolved to the GitHub
    Pages copy) or a full https URL."""
    base = "https://shammun.github.io/shammunul-fastai-notes/notebooks/"
    if not path.startswith("http"):
        path = base + path
    return HTML(
        f'<iframe src="{path}" loading="lazy" allowfullscreen '
        f'style="width:100%;height:{height};border:1px solid #dde5f2;border-radius:12px;'
        f'box-shadow:0 8px 24px rgba(108,92,231,.12);background:#fff;"></iframe>'
        '<script>addEventListener("message",function(e){'
        'if(e.data&&(e.data.type==="ae-frame-height"||e.data.type==="ce-frame-height")&&e.data.height>50){'
        'var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){'
        'if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>'
    )

In [5]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. Loads interactive_viz/ae_pipeline_explorer.html (hosted on
# GitHub Pages) in a full-width iframe that auto-fits its height. End-to-end autoencoder pipeline: Input -> Encoder -> Bottleneck -> Decoder -> Output.
# Requires the show_viz() setup cell above.
# ============================================================================
show_viz("interactive_viz/ae_pipeline_explorer.html", height="780px")

# Part 1: Setup and Imports

---

First, let's import all the libraries we need.

In [6]:
# =====================================================
# STANDARD PYTHON LIBRARIES
# =====================================================

import pickle          # For loading/saving Python objects
import gzip            # For compressed files
import math            # Mathematical functions
import os              # Operating system interface
import time            # Timing operations
import shutil          # File operations

# =====================================================
# DATA SCIENCE LIBRARIES
# =====================================================

import numpy as np                    # Numerical computing
import matplotlib as mpl              # Plotting configuration
import matplotlib.pyplot as plt       # Creating plots

# =====================================================
# PYTORCH LIBRARIES
# =====================================================

import torch                          # Main PyTorch library
from torch import tensor, nn, optim   # Tensors, neural networks, optimizers
from torch.utils.data import DataLoader, default_collate  # Data loading utilities
import torch.nn.functional as F       # Functional operations (loss functions, etc.)
import torchvision.transforms.functional as TF  # Image transformations

# =====================================================
# HUGGING FACE DATASETS
# =====================================================

# datasets is a library from Hugging Face that provides easy access
# to many popular datasets, including Fashion MNIST
from datasets import load_dataset, load_dataset_builder

# =====================================================
# UTILITY LIBRARIES
# =====================================================

from pathlib import Path              # Modern path handling
from operator import attrgetter, itemgetter  # Efficient attribute/item access
from functools import partial         # Partial function application
from collections.abc import Mapping   # For type checking dictionaries

# =====================================================
# FASTAI UTILITIES (from previous notebooks)
# =====================================================

# These are helper functions from the fastai library and our miniai module
import fastcore.all as fc
from fastcore.test import test_close  # Testing utility
from fastprogress import progress_bar, master_bar  # Progress bars

In [7]:
# =====================================================
# CONFIGURATION AND SETTINGS
# =====================================================

# Set how PyTorch displays tensors:
#   precision=2     - Show 2 decimal places
#   linewidth=140   - Wider lines before wrapping
#   sci_mode=False  - Don't use scientific notation
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)

# Set random seed for reproducibility
# This ensures we get the same "random" results each time
torch.manual_seed(1)

# Use grayscale colormap for images (Fashion MNIST is black and white)
mpl.rcParams['image.cmap'] = 'gray'

# Disable warning messages from the datasets library
import logging
logging.disable(logging.WARNING)

In [8]:
# =====================================================
# DEVICE SELECTION (GPU IF AVAILABLE)
# =====================================================

# Choose the best available device:
#   - MPS: Apple Silicon GPU (M1/M2 Macs)
#   - CUDA: NVIDIA GPU
#   - CPU: Fallback if no GPU available

if torch.backends.mps.is_available():
    def_device = 'mps'
elif torch.cuda.is_available():
    def_device = 'cuda'
else:
    def_device = 'cpu'

print(f"Using device: {def_device}")

Using device: cuda


In [9]:
# =====================================================
# HELPER FUNCTIONS (from previous notebooks)
# =====================================================

def to_device(x, device=def_device):
    """
    Move a tensor or collection of tensors to the specified device.

    Arguments:
        x      - A tensor, dictionary, or iterable of tensors
        device - The target device ('cuda', 'mps', or 'cpu')

    Returns:
        The same structure with all tensors moved to the device
    """
    if isinstance(x, torch.Tensor):
        return x.to(device)
    if isinstance(x, Mapping):
        return {k: v.to(device) for k, v in x.items()}
    return type(x)(to_device(o, device) for o in x)


def conv(ni, nf, ks=3, stride=2, act=True):
    """
    Create a convolution layer with optional ReLU activation.

    This is the same helper function from the convolutions notebook.

    Arguments:
        ni     - Number of input channels
        nf     - Number of output channels (filters)
        ks     - Kernel size (default 3 for 3x3)
        stride - Stride (default 2 to halve spatial size)
        act    - Whether to add ReLU activation (default True)

    Returns:
        A Conv2d layer, optionally wrapped with ReLU
    """
    res = nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2)
    if act:
        res = nn.Sequential(res, nn.ReLU())
    return res


def show_image(im, ax=None, figsize=(3,3), title=None, noframe=True, cmap=None):
    """Display a single image."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    if hasattr(im, 'cpu'):
        im = im.detach().cpu()
    if hasattr(im, 'numpy'):
        im = im.numpy()
    if im.ndim == 3 and im.shape[0] in [1, 3]:
        im = im.transpose(1, 2, 0)
    if im.ndim == 3 and im.shape[2] == 1:
        im = im[:, :, 0]
    ax.imshow(im, cmap=cmap)
    if title is not None:
        ax.set_title(title)
    if noframe:
        ax.axis('off')
    return ax


def show_images(ims, nrows=1, ncols=None, titles=None, figsize=None, imsize=3):
    """Display multiple images in a grid."""
    if ncols is None:
        ncols = len(ims)
    if figsize is None:
        figsize = (ncols * imsize, nrows * imsize)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows * ncols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    for i, (im, ax) in enumerate(zip(ims, axes)):
        title = titles[i] if titles else None
        show_image(im, ax=ax, title=title)
    plt.tight_layout()
    return fig

---

# Part 2: Loading the Fashion MNIST Dataset

---

## What is Fashion MNIST?

**Fashion MNIST** is a dataset of clothing images, designed as a drop-in replacement for the classic MNIST digit dataset. It's more challenging while having the same format:

- **60,000 training images** and **10,000 test images**
- **28 × 28 pixels** each, grayscale
- **10 classes** of clothing items

| Label | Description |
|-------|-------------|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

---

In [10]:
# =====================================================
# LOADING FASHION MNIST WITH HUGGING FACE DATASETS
# =====================================================

# Define the column names in the dataset
x = 'image'   # The column containing images
y = 'label'   # The column containing labels (0-9)

# Name of the dataset on Hugging Face
name = "fashion_mnist"

# load_dataset downloads and caches the dataset
# ignore_verifications=True skips hash verification (faster loading)
#
# Returns a DatasetDict with 'train' and 'test' splits
dsd = load_dataset(name, ignore_verifications=True)

print(f"Dataset loaded: {name}")
print(f"Splits available: {list(dsd.keys())}")
print(f"Training samples: {len(dsd['train'])}")
print(f"Test samples: {len(dsd['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/9.02k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/fashion_mnist@531be5e2ccc9dba0c201ad3ae567a4f3d16ecdd2/.huggingface.yaml'. Repository id must be 'namespace/name', got 'fashion_mnist'.

In [ ]:
# Let's look at what one sample looks like
sample = dsd['train'][0]
print("Sample structure:")
print(f"  Keys: {sample.keys()}")
print(f"  Image type: {type(sample['image'])}")
print(f"  Label: {sample['label']}")

---

## Transforming the Data

The images come as PIL Image objects. We need to convert them to PyTorch tensors for training.

In [ ]:
# =====================================================
# DEFINING THE TRANSFORMATION
# =====================================================

# The @inplace decorator modifies the input dictionary directly
# instead of returning a new one. This saves memory.
#
# If you don't have the inplace decorator, you can define it:
def inplace(f):
    """Decorator that makes a function modify its argument in place."""
    def wrapper(b):
        f(b)
        return b
    return wrapper

@inplace
def transformi(b):
    """
    Transform a batch of data by converting images to tensors.

    Arguments:
        b - A dictionary with 'image' key containing PIL Images

    Modifies b in place:
        - Converts each PIL Image to a PyTorch tensor
        - Tensor values are scaled to [0, 1] range
        - Shape becomes (1, 28, 28) - 1 channel, 28x28 pixels

    TF.to_tensor does several things:
        1. Converts PIL Image to PyTorch tensor
        2. Rearranges from (H, W, C) to (C, H, W)
        3. Scales pixel values from [0, 255] to [0.0, 1.0]
    """
    b[x] = [TF.to_tensor(o) for o in b[x]]

In [ ]:
# =====================================================
# APPLYING THE TRANSFORMATION
# =====================================================

# Batch size - how many samples to process at once
bs = 256

# with_transform applies our transformation lazily (when data is accessed)
# This is memory-efficient: images are converted only when needed
tds = dsd.with_transform(transformi)

print("Transformed dataset created.")
print("Transformation will be applied when data is accessed.")

In [ ]:
# =====================================================
# EXAMINING A TRANSFORMED SAMPLE
# =====================================================

# Get the training split
ds = tds['train']

# Get the first image (now it's a tensor!)
img = ds[0]['image']

print(f"Image type: {type(img)}")
print(f"Image shape: {img.shape}")
print(f"  - {img.shape[0]} channel (grayscale)")
print(f"  - {img.shape[1]}x{img.shape[2]} pixels")
print(f"Pixel value range: {img.min():.2f} to {img.max():.2f}")

# Display the image
show_image(img, figsize=(2, 2))

---

## Creating Data Loaders

Data loaders batch the data and handle shuffling for training.

In [ ]:
# =====================================================
# COLLATE FUNCTION
# =====================================================

# A collate function takes a list of samples and combines them into a batch.
#
# The Hugging Face dataset returns dictionaries like:
#   {'image': tensor, 'label': int}
#
# We need to:
#   1. Stack all images into a single tensor
#   2. Stack all labels into a single tensor
#   3. Move everything to the GPU

def collate_dict(ds):
    """
    Create a collate function for dictionary-based datasets.

    Returns a function that:
        1. Extracts 'image' and 'label' from each sample
        2. Stacks them into batch tensors
    """
    def _collate(batch):
        # Stack all images: list of (1,28,28) -> (batch_size, 1, 28, 28)
        images = torch.stack([b['image'] for b in batch])
        # Stack all labels: list of ints -> (batch_size,)
        labels = torch.tensor([b['label'] for b in batch])
        return images, labels
    return _collate

# Create the collate function for our dataset
cf = collate_dict(ds)

### Understanding the Collate Function: Closure Pattern

#### What happens with `cf = collate_dict(ds)`?

`collate_dict` is **already defined** — calling it doesn't *create* the function. Instead, it **calls** `collate_dict`, which **returns the inner function `_collate`**.

So after this line, `cf` **is** the `_collate` function itself — it's a **function object**, not a computed result.

```python
cf = collate_dict(ds)
# cf IS _collate — it's a callable function, not a data result
```

> **Note:** In the current implementation, the `ds` argument isn't actually used inside `_collate`. It's likely there for future flexibility or was used in an earlier version of the code.

---

#### What happens with `cf(b)` inside `collate_`?

Since `cf` *is* the `_collate` function, calling `cf(b)` is identical to calling `_collate(b)`:

1. It takes a **batch** `b` — a list of dictionaries like `[{'image': tensor, 'label': int}, ...]`
2. It **stacks** all images into a single tensor of shape `(batch_size, 1, 28, 28)`
3. It **stacks** all labels into a single tensor of shape `(batch_size,)`
4. It returns the tuple `(images, labels)`

The full chain inside `collate_(b)` works like this:

```
b (list of dicts)
    │
    ▼
cf(b)  →  _collate(b)  →  (stacked_images, stacked_labels)
    │
    ▼
to_device(...)  →  moves both tensors to GPU
    │
    ▼
returns (images_on_gpu, labels_on_gpu)
```

---

#### Why This Pattern? — The Factory Function

This is a **factory function** (also called a **closure**) — a function that **builds and returns another function**.

**Why is it useful here?**

PyTorch's `DataLoader` expects a collate function with the signature `collate(batch)` — just **one argument**. By wrapping the logic inside a factory:

- You can **bake in configuration** (like dataset structure, column names, etc.) at creation time
- You still hand the `DataLoader` a **clean, single-argument function** that it knows how to call

```python
# Factory creates a configured function
cf = collate_dict(ds)        # returns _collate, configured for dict-based datasets

# That function is then used inside another function
def collate_(b):
    return to_device(cf(b))  # stack into batch → move to GPU

# DataLoader only sees a simple collate_(batch) signature
dl = DataLoader(dataset, collate_fn=collate_)
```

This is a common and elegant pattern in Python for **composing behavior** while keeping interfaces clean.

In [ ]:
# =====================================================
# DATA LOADERS WITH DEVICE TRANSFER
# =====================================================

def collate_(b):
    """
    Collate function that also moves data to the device (GPU).

    Combines cf (which stacks samples) with to_device (which moves to GPU).
    """
    return to_device(cf(b))


def data_loaders(dsd, bs, **kwargs):
    """
    Create DataLoaders for all splits in a dataset dict.

    Arguments:
        dsd    - A DatasetDict with 'train', 'test', etc.
        bs     - Batch size
        kwargs - Additional arguments for DataLoader

    Returns:
        A dictionary of DataLoaders, one per split
    """
    return {k: DataLoader(v, bs, **kwargs) for k, v in dsd.items()}

# Understanding `data_loaders`: Creating DataLoaders from a DatasetDict

## What does this function do?

A Hugging Face `DatasetDict` contains **multiple splits** of your data — typically `'train'` and `'test'`. This function takes that entire dict and creates a **PyTorch `DataLoader`** for each split in one go.

```python
def data_loaders(dsd, bs, **kwargs):
    return {k: DataLoader(v, bs, **kwargs) for k, v in dsd.items()}
```

---

## Breaking it down step by step

### 1. `dsd.items()` — Iterating over the DatasetDict

`dsd` is a Hugging Face `DatasetDict`, which behaves like a Python dictionary:

```python
dsd = {
    'train': <Dataset with 60000 samples>,
    'test':  <Dataset with 10000 samples>
}
```

Calling `dsd.items()` gives you key-value pairs:

| `k` (key) | `v` (value) |
|---|---|
| `'train'` | The training `Dataset` object |
| `'test'` | The test `Dataset` object |

### 2. `DataLoader(v, bs, **kwargs)` — Wrapping each dataset

For each split, a PyTorch `DataLoader` is created with:

- `v` — the dataset for that split
- `bs` — the batch size
- `**kwargs` — any additional arguments (e.g., `collate_fn=collate_`, `shuffle=True`, `num_workers=4`)

### 3. Dictionary comprehension — Collecting the results

The `{k: ... for k, v in ...}` syntax builds a **new dictionary** that maps each split name to its corresponding `DataLoader`.

---

## What the output looks like

```python
dls = data_loaders(dsd, bs=64, collate_fn=collate_)

# dls is now:
# {
#     'train': DataLoader(train_dataset, batch_size=64, collate_fn=collate_),
#     'test':  DataLoader(test_dataset,  batch_size=64, collate_fn=collate_)
# }

# Access them like:
train_dl = dls['train']
test_dl  = dls['test']
```

---

## Why `**kwargs`?

The `**kwargs` pattern lets you **pass through any extra arguments** to `DataLoader` without hardcoding them. This keeps the function flexible:

```python
# Minimal usage
dls = data_loaders(dsd, bs=64)

# With extra options
dls = data_loaders(dsd, bs=64, collate_fn=collate_, num_workers=4, shuffle=True)
```

All the keyword arguments after `bs` get collected into `kwargs` and forwarded directly to each `DataLoader` constructor via `**kwargs`.

---

## Summary

This is a **convenience wrapper** that avoids writing repetitive code like:

```python
# Without the helper — repetitive
train_dl = DataLoader(dsd['train'], 64, collate_fn=collate_)
test_dl  = DataLoader(dsd['test'],  64, collate_fn=collate_)
```

Instead, one line handles all splits:

```python
# With the helper — clean and scalable
dls = data_loaders(dsd, 64, collate_fn=collate_)
```

If your `DatasetDict` had more splits (e.g., `'train'`, `'validation'`, `'test'`), this function would automatically create a `DataLoader` for each one — no extra code needed.

In [ ]:
# =====================================================
# CREATE THE DATA LOADERS
# =====================================================

# Create loaders for train and test splits
dls = data_loaders(tds, bs, collate_fn=collate_)

print(f"Data loaders created: {list(dls.keys())}")
print(f"Batch size: {bs}")

In [ ]:
# =====================================================
# GET CONVENIENT REFERENCES
# =====================================================

# Shortcuts for training and validation (test) loaders
dt = dls['train']   # Training data loader
dv = dls['test']    # Validation/test data loader

# Get one batch to inspect
# next(iter(dt)) gets the first batch from the training loader
xb, yb = next(iter(dt))

print(f"Batch shapes:")
print(f"  Images (xb): {xb.shape}")
print(f"    - {xb.shape[0]} images in batch")
print(f"    - {xb.shape[1]} channel")
print(f"    - {xb.shape[2]}x{xb.shape[3]} pixels")
print(f"  Labels (yb): {yb.shape}")
print(f"  Device: {xb.device}")

In [ ]:
# =====================================================
# CLASS LABELS
# =====================================================

# The dataset has human-readable names for each class
# ds.features[y] contains metadata about the label column
# .names gives us the list of class names

labels = ds.features[y].names

print("Fashion MNIST Classes:")
for i, label in enumerate(labels):
    print(f"  {i}: {label}")

In [ ]:
# =====================================================
# VISUALIZING A BATCH
# =====================================================

# itemgetter creates a function that extracts items by index
# itemgetter(0, 3, 5) returns a function that returns (seq[0], seq[3], seq[5])
#
# Here we use it to get the label names for our batch
# yb[:16] are the first 16 label indices (e.g., [9, 0, 0, 3, ...])
# We want to convert these to names like ['Ankle boot', 'T-shirt/top', ...]

# *yb[:16] unpacks the tensor to individual arguments
lbl_getter = itemgetter(*yb[:16].tolist())
titles = lbl_getter(labels)

print(f"First 16 labels: {yb[:16].tolist()}")
print(f"Label names: {titles}")

#### Understanding `itemgetter` for Label Lookup

#### The Problem We're Solving

We have:

- `yb[:16]` — a tensor of **16 label indices**, e.g., `[9, 0, 0, 3, 7, ...]`
- `labels` — a list/tuple of **label names**, e.g., `['T-shirt/top', 'Trouser', 'Pullover', ..., 'Ankle boot']`

We want to convert those indices into human-readable names like `['Ankle boot', 'T-shirt/top', 'T-shirt/top', 'Dress', ...]`.

---

#### Line 1: `lbl_getter = itemgetter(*yb[:16].tolist())`

This line does **three things** — let's unpack them from the inside out.

#### Step 1: `yb[:16].tolist()`

Converts the first 16 label indices from a PyTorch tensor to a plain Python list:

```python
yb[:16]           # tensor([9, 0, 0, 3, 7, ...])
yb[:16].tolist()  # [9, 0, 0, 3, 7, ...]
```

#### Step 2: `*` (unpacking operator)

The `*` **unpacks** the list into separate arguments:

```python
itemgetter(*[9, 0, 0, 3, 7])
# is equivalent to:
itemgetter(9, 0, 0, 3, 7)
```

Without `*`, you'd pass a single list as one argument — `itemgetter` would not know what to do with it. With `*`, each index becomes a **separate argument**.

#### Step 3: `itemgetter(9, 0, 0, 3, 7, ...)`

`itemgetter` from Python's `operator` module **creates a function** that, when called on a sequence, extracts items at the specified indices.

```python
lbl_getter = itemgetter(9, 0, 0, 3, 7)
# lbl_getter is now a FUNCTION that does:
#   given a sequence s → return (s[9], s[0], s[0], s[3], s[7])
```

> **Key insight:** `itemgetter` doesn't fetch anything yet — it **builds a reusable lookup function**.

---

#### Line 2: `titles = lbl_getter(labels)`

Now we **call** that function on `labels` (the list of label names):

```python
labels = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
          'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

titles = lbl_getter(labels)
# Executes: (labels[9], labels[0], labels[0], labels[3], labels[7], ...)
# Returns:  ('Ankle boot', 'T-shirt/top', 'T-shirt/top', 'Dress', 'Sneaker', ...)
```

---

#### The Full Flow Visualized

```
yb[:16]                    →  tensor([9, 0, 0, 3, 7, ...])
    │
    ▼  .tolist()
[9, 0, 0, 3, 7, ...]      →  plain Python list
    │
    ▼  * (unpack)
itemgetter(9, 0, 0, 3, 7)  →  creates a FUNCTION
    │
    ▼  call with labels
(labels[9], labels[0], ...) →  ('Ankle boot', 'T-shirt/top', ...)
```

---

## Why not just use a list comprehension?

You absolutely could write this as:

```python
titles = [labels[i] for i in yb[:16].tolist()]
```

Both approaches work. The `itemgetter` version is:

- **Slightly faster** for large sequences (implemented in C)
- **Reusable** — you can call `lbl_getter` on different label lists without rebuilding
- **A common pattern** in fast.ai code, which favors functional programming style

The list comprehension is arguably more readable if you're not used to `itemgetter`, but both produce the same result.

In [ ]:
# =====================================================
# DISPLAY SAMPLE IMAGES
# =====================================================

mpl.rcParams['figure.dpi'] = 70

# Show the first 16 images with their labels
# Need to move to CPU for matplotlib
show_images(xb[:16].cpu(), nrows=2, ncols=8, titles=titles, imsize=1.7)

---

# Part 3: Warmup - CNN Classification

---

Before building our autoencoder, let's verify our setup works by training a simple CNN classifier. This is the same architecture from the convolutions notebook.

**Why do this warmup?**
- Confirms our data pipeline is working
- Verifies GPU is being used correctly
- Reviews the CNN concepts before we modify them for autoencoders

In [ ]:
# =====================================================
# TRAINING HYPERPARAMETERS
# =====================================================

bs = 256    # Batch size (already set, but good to be explicit)
lr = 0.4    # Learning rate for SGD optimizer

In [ ]:
# =====================================================
# CNN CLASSIFIER ARCHITECTURE
# =====================================================

# This is the same CNN from the convolutions notebook:
#   - 5 convolutional layers with stride=2
#   - Each layer halves the spatial dimensions
#   - Final layer outputs 10 values (one per class)
#
# Size progression:
#   28x28 -> 14x14 -> 7x7 -> 4x4 -> 2x2 -> 1x1

cnn = nn.Sequential(
    conv(1, 4),              # 28x28 -> 14x14, 4 channels
    conv(4, 8),              # 14x14 -> 7x7,   8 channels
    conv(8, 16),             # 7x7   -> 4x4,   16 channels
    conv(16, 16),            # 4x4   -> 2x2,   16 channels
    conv(16, 10, act=False), # 2x2   -> 1x1,   10 channels (no ReLU!)
    nn.Flatten()             # (batch, 10, 1, 1) -> (batch, 10)
).to(def_device)

print("CNN Classifier Architecture:")
print(cnn)

# How `conv(1, 4)` Transforms 28×28 → 14×14 with 4 Channels

## The `conv` Function

Here's the `conv` function from the notebook:

```python
def conv(ni, nf, ks=3, stride=2, act=True):
    res = nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2)
    if act:
        res = nn.Sequential(res, nn.ReLU())
    return res
```

When we call `conv(1, 4)`, we get these parameters:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `ni` | 1 | Input channels (grayscale image) |
| `nf` | 4 | Output channels (4 filters) |
| `ks` | 3 | Kernel size (3×3) |
| `stride` | 2 | Step size (default) |
| `padding` | 1 | Computed as `ks//2 = 3//2 = 1` |

---

## Two Independent Transformations

The convolution performs two separate operations simultaneously:

1. **Spatial reduction** (28 → 14) — controlled by `stride`
2. **Channel expansion** (1 → 4) — controlled by `nf`

---

## 1. Spatial Reduction: 28 → 14

### Why Does `stride=2` Halve the Size?

With `stride=2`, the kernel jumps 2 pixels at a time instead of 1:

```
stride=1: ● ● ● ● ● ● ● ●  →  kernel visits every position
stride=2: ●   ●   ●   ●    →  kernel skips every other position
```

### The Output Size Formula

$$\text{output size} = \left\lfloor \frac{\text{input} + 2 \times \text{padding} - \text{kernel}}{\text{stride}} \right\rfloor + 1$$

Plugging in our values:

$$\text{output size} = \left\lfloor \frac{28 + 2(1) - 3}{2} \right\rfloor + 1 = \left\lfloor \frac{27}{2} \right\rfloor + 1 = 13 + 1 = 14$$

### Visual Intuition

Think of it like sampling every other pixel:

```
Input row (28 pixels):
[0][1][2][3][4][5][6][7][8][9]...[27]
 ↓     ↓     ↓     ↓     ↓
[0]   [2]   [4]   [6]   [8]  ...  → 14 output positions
```

---

## 2. Channel Expansion: 1 → 4

### What Are Channels?

- **Input channel**: The grayscale image has 1 channel (pixel intensity)
- **Output channels**: Each filter produces one output channel (feature map)

### How Filters Create Channels

Each filter is a learnable 3×3 weight matrix. With 4 filters, we get 4 output channels:

```
                    Filter 0 (3×3) ──→ Channel 0 (14×14)
                         │
Input (1×28×28) ──→ Filter 1 (3×3) ──→ Channel 1 (14×14)
                         │
                    Filter 2 (3×3) ──→ Channel 2 (14×14)
                         │
                    Filter 3 (3×3) ──→ Channel 3 (14×14)
```

### What Do Filters Learn?

During training, each filter learns to detect different features:

| Filter | Might Learn to Detect |
|--------|----------------------|
| Filter 0 | Horizontal edges |
| Filter 1 | Vertical edges |
| Filter 2 | Corners or curves |
| Filter 3 | Textures or patterns |

The network discovers these features automatically through backpropagation.

---

## Visual Summary

```
        INPUT                          OUTPUT
     ┌─────────┐                    ┌───────┐
     │         │                    │ Ch 0  │ 14×14
     │  28×28  │     conv(1,4)      ├───────┤
     │         │  ─────────────→    │ Ch 1  │ 14×14
     │ 1 chan  │   stride=2         ├───────┤
     │         │   4 filters        │ Ch 2  │ 14×14
     └─────────┘                    ├───────┤
                                    │ Ch 3  │ 14×14
                                    └───────┘

      Shape: (1, 28, 28)            Shape: (4, 14, 14)
      Values: 784                   Values: 784
```

Interestingly, the total number of values stays the same (784) in this first layer — we've traded spatial resolution for richer feature representation.

---

## The Full CNN Progression

Here's how the entire CNN classifier transforms the input:

| Layer | Operation | Output Shape | Spatial Size | Channels | Total Values |
|-------|-----------|--------------|--------------|----------|--------------|
| Input | — | (1, 28, 28) | 28×28 | 1 | 784 |
| `conv(1, 4)` | stride=2, 4 filters | (4, 14, 14) | 14×14 | 4 | 784 |
| `conv(4, 8)` | stride=2, 8 filters | (8, 7, 7) | 7×7 | 8 | 392 |
| `conv(8, 16)` | stride=2, 16 filters | (16, 4, 4) | 4×4 | 16 | 256 |
| `conv(16, 16)` | stride=2, 16 filters | (16, 2, 2) | 2×2 | 16 | 64 |
| `conv(16, 10)` | stride=2, 10 filters | (10, 1, 1) | 1×1 | 10 | 10 |
| `Flatten()` | reshape | (10,) | — | — | 10 |

### The Pattern

As we go deeper:
- **Spatial size decreases**: 28 → 14 → 7 → 4 → 2 → 1
- **Channels increase**: 1 → 4 → 8 → 16 → 16 → 10
- **Information compresses**: 784 → 784 → 392 → 256 → 64 → 10

The network learns to extract increasingly abstract features while compressing the representation down to 10 values — one score per Fashion MNIST class.

---

## Key Takeaways

1. **Stride controls spatial reduction**: `stride=2` halves the height and width
2. **Number of filters controls output channels**: `nf=4` means 4 output channels
3. **These are independent**: You can have any combination of stride and filter count
4. **Padding preserves information**: `padding=ks//2` prevents excessive shrinking at borders
5. **CNNs trade space for depth**: Smaller spatial size, more channels = richer features

### 🔍 Visualize It: Stride and Filters

Drag the orange kernel over the input, toggle **stride = 1 vs 2**, and watch the output grid fill in. The formula on the right resolves live, and the lower panel shows how 4 filters produce 4 output channels.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. Loads interactive_viz/conv_stride_lab.html (hosted on
# GitHub Pages) in a full-width iframe that auto-fits its height. How stride changes a conv layer's output size.
# Requires the show_viz() setup cell above.
# ============================================================================
show_viz("interactive_viz/conv_stride_lab.html", height="820px")

In [ ]:
# =====================================================
# TRAINING FUNCTION FOR CLASSIFICATION
# =====================================================

def accuracy(preds, targets):
    """Calculate accuracy: fraction of correct predictions."""
    return (preds.argmax(dim=1) == targets).float().mean()

def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    """
    Train a classification model.

    Arguments:
        epochs   - Number of complete passes through the data
        model    - The neural network to train
        loss_func - Function to compute loss (e.g., cross_entropy)
        opt      - Optimizer (e.g., SGD)
        train_dl - Training data loader
        valid_dl - Validation data loader

    Returns:
        Final (loss, accuracy) on validation set
    """
    for epoch in range(epochs):
        # Training phase
        model.train()
        for xb, yb in train_dl:
            # Forward pass
            preds = model(xb)
            loss = loss_func(preds, yb)

            # Backward pass
            loss.backward()
            opt.step()
            opt.zero_grad()

        # Validation phase
        model.eval()
        with torch.no_grad():
            tot_loss, tot_acc, count = 0., 0., 0
            for xb, yb in valid_dl:
                preds = model(xb)
                n = len(xb)
                count += n
                tot_loss += loss_func(preds, yb).item() * n
                tot_acc += accuracy(preds, yb).item() * n

        print(f"Epoch {epoch}: loss={tot_loss/count:.4f}, accuracy={tot_acc/count:.4f}")

    return tot_loss/count, tot_acc/count

# Understanding the Training Function for Classification

## Overview

This code defines two functions:
1. **`accuracy`** — Measures how many predictions are correct
2. **`fit`** — The main training loop that teaches the neural network

---

## The `accuracy` Function

```python
def accuracy(preds, targets):
    """Calculate accuracy: fraction of correct predictions."""
    return (preds.argmax(dim=1) == targets).float().mean()
```

### What It Does

Calculates the fraction of correct predictions (e.g., 0.85 means 85% correct).

### Step-by-Step Breakdown

| Step | Code | Example | Result |
|------|------|---------|--------|
| 1. Get predicted class | `preds.argmax(dim=1)` | `[[0.1, 0.9, 0.0], [0.8, 0.1, 0.1]]` | `[1, 0]` |
| 2. Compare to targets | `== targets` | `[1, 0] == [1, 2]` | `[True, False]` |
| 3. Convert to numbers | `.float()` | `[True, False]` | `[1.0, 0.0]` |
| 4. Calculate average | `.mean()` | `[1.0, 0.0]` | `0.5` (50%) |

### Visual Example

```
Predictions (raw scores for 3 classes):
  Image 0: [0.1, 0.9, 0.0] → argmax → Class 1 ✓ (target was 1)
  Image 1: [0.8, 0.1, 0.1] → argmax → Class 0 ✗ (target was 2)

Accuracy = 1 correct / 2 total = 0.5 (50%)
```

---

## The `fit` Function

```python
def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
```

### Arguments Explained

| Argument | What It Is | Example |
|----------|-----------|---------|
| `epochs` | Number of complete passes through all data | `5` |
| `model` | The neural network to train | `cnn` |
| `loss_func` | Function measuring prediction error | `F.cross_entropy` |
| `opt` | Optimizer that updates weights | `optim.SGD(...)` |
| `train_dl` | DataLoader with training data | Batches of images + labels |
| `valid_dl` | DataLoader with validation data | Batches for evaluation |

---

## The Training Loop Explained

### High-Level Structure

```
For each epoch:
    1. TRAINING PHASE   → Learn from training data
    2. VALIDATION PHASE → Check performance on unseen data
    3. Print results
```

---

### Phase 1: Training

```python
model.train()
for xb, yb in train_dl:
    # Forward pass
    preds = model(xb)
    loss = loss_func(preds, yb)
    
    # Backward pass
    loss.backward()
    opt.step()
    opt.zero_grad()
```

#### What Each Line Does

| Line | Purpose | Analogy |
|------|---------|---------|
| `model.train()` | Enable training mode (activates dropout, etc.) | "Get ready to learn" |
| `for xb, yb in train_dl` | Loop through batches of images (`xb`) and labels (`yb`) | "Go through homework problems" |
| `preds = model(xb)` | **Forward pass**: Make predictions | "Attempt the problem" |
| `loss = loss_func(preds, yb)` | Calculate how wrong we were | "Check the answer key" |
| `loss.backward()` | **Backward pass**: Compute gradients | "Figure out what went wrong" |
| `opt.step()` | Update weights using gradients | "Adjust your understanding" |
| `opt.zero_grad()` | Reset gradients for next batch | "Clear your scratch paper" |

#### Visual Flow

```
    ┌─────────────┐
    │ Input Batch │ (xb: 256 images)
    │   28×28×1   │
    └──────┬──────┘
           │
           ▼ Forward Pass
    ┌─────────────┐
    │    Model    │ (CNN)
    └──────┬──────┘
           │
           ▼
    ┌─────────────┐
    │ Predictions │ (preds: 256 × 10 scores)
    └──────┬──────┘
           │
           ▼ Compare with labels (yb)
    ┌─────────────┐
    │    Loss     │ (single number: how wrong?)
    └──────┬──────┘
           │
           ▼ Backward Pass
    ┌─────────────┐
    │  Gradients  │ (which direction to adjust?)
    └──────┬──────┘
           │
           ▼ Optimizer Step
    ┌─────────────┐
    │Update Weights│ (make model better)
    └─────────────┘
```

---

### Phase 2: Validation

```python
model.eval()
with torch.no_grad():
    tot_loss, tot_acc, count = 0., 0., 0
    for xb, yb in valid_dl:
        preds = model(xb)
        n = len(xb)
        count += n
        tot_loss += loss_func(preds, yb).item() * n
        tot_acc += accuracy(preds, yb).item() * n
```

#### Key Differences from Training

| Aspect | Training | Validation |
|--------|----------|------------|
| Mode | `model.train()` | `model.eval()` |
| Gradients | Computed | `torch.no_grad()` — disabled |
| Weight updates | Yes | No |
| Purpose | Learn | Evaluate |

#### Why These Differences?

1. **`model.eval()`**: Disables dropout and uses running statistics for batch normalization — gives consistent predictions

2. **`torch.no_grad()`**: Saves memory and computation since we don't need gradients for evaluation

3. **No `backward()` or `opt.step()`**: We're just measuring, not learning

#### Accumulating Statistics

```python
tot_loss += loss_func(preds, yb).item() * n
tot_acc += accuracy(preds, yb).item() * n
count += n
```

We multiply by `n` (batch size) because we want weighted averages:

```
Batch 1: 256 images, 85% accuracy → contributes 256 × 0.85 = 217.6
Batch 2: 256 images, 90% accuracy → contributes 256 × 0.90 = 230.4
...
Final: (217.6 + 230.4 + ...) / total_images = weighted average
```

---

## Complete Flow Diagram

```
                    ┌─────────────────────────────────────┐
                    │           EPOCH 0                   │
                    └─────────────────────────────────────┘
                                    │
          ┌─────────────────────────┴─────────────────────────┐
          ▼                                                   ▼
┌─────────────────────┐                         ┌─────────────────────┐
│   TRAINING PHASE    │                         │  VALIDATION PHASE   │
│                     │                         │                     │
│ For each batch:     │                         │ For each batch:     │
│  • Forward pass     │                         │  • Forward pass     │
│  • Compute loss     │                         │  • Compute loss     │
│  • Backward pass    │                         │  • Compute accuracy │
│  • Update weights   │                         │  • Accumulate stats │
│                     │                         │                     │
│ (Learning happens!) │                         │ (No learning!)      │
└─────────────────────┘                         └──────────┬──────────┘
                                                           │
                                                           ▼
                                                ┌─────────────────────┐
                                                │ Print: loss=0.52,   │
                                                │ accuracy=0.82       │
                                                └─────────────────────┘
                                                           │
                                                           ▼
                                                      Epoch 1...
```

---

## Why Separate Training and Validation?

| Training Data | Validation Data |
|--------------|-----------------|
| Model learns from this | Model never learns from this |
| Can memorize patterns | Tests generalization |
| Loss goes down (expected) | Loss should also go down |

**If training loss ↓ but validation loss ↑** → Overfitting! Model memorized training data but can't generalize.

---

## Example Output

```
Epoch 0: loss=0.8234, accuracy=0.7156
Epoch 1: loss=0.5621, accuracy=0.8012
Epoch 2: loss=0.4892, accuracy=0.8298
Epoch 3: loss=0.4501, accuracy=0.8445
Epoch 4: loss=0.4289, accuracy=0.8523
```

Each epoch:
- Loss decreases → Model is making smaller errors
- Accuracy increases → Model is getting more predictions right

---

## Key Takeaways

1. **Training loop = Forward → Loss → Backward → Update** (repeat for all batches)

2. **Validation = Forward → Measure** (no learning, just evaluation)

3. **`model.train()` vs `model.eval()`** — Different behaviors for training vs inference

4. **`torch.no_grad()`** — Saves memory during validation

5. **Epochs** — One epoch = one complete pass through all training data

# Understanding Validation Accumulation: `.item()` and `* n`

## The Two Questions

```python
tot_loss += loss_func(preds, yb).item() * n
tot_acc  += accuracy(preds, yb).item() * n
```

### 1. Why `.item()`?

`loss_func(preds, yb)` returns a **PyTorch tensor** (a single-element tensor on the GPU), not a plain Python number.

```python
loss = loss_func(preds, yb)
print(type(loss))   # <class 'torch.Tensor'>
print(loss)         # tensor(0.4523, device='cuda:0')
```

**Problem:** If you accumulate tensors in a loop, PyTorch **keeps the entire computation graph** in memory because it thinks you might want to call `.backward()` later. Over hundreds of batches, this **leaks GPU memory** massively.

`.item()` extracts the value as a **plain Python float**, breaking it free from the computation graph:

```python
loss.item()         # 0.4523 (plain float, no graph, no GPU memory held)
```

> **Rule of thumb:** During validation (where you don't need gradients), always use `.item()` when accumulating scalar values to avoid memory leaks.

---

### 2. Why multiply by `n`?

This is about computing a **correct weighted average** when the last batch might be smaller.

#### The Problem

`loss_func` and `accuracy` both return **mean values for that batch**. But not all batches are the same size — the **last batch** is often smaller:

```
Batch 1: 64 samples → mean_loss = 0.45
Batch 2: 64 samples → mean_loss = 0.42
Batch 3: 64 samples → mean_loss = 0.38
Batch 4: 24 samples → mean_loss = 0.50  ← last batch, only 24 samples!
```

#### Wrong approach — simple average of means

```python
# WRONG: treats all batches equally
avg_loss = (0.45 + 0.42 + 0.38 + 0.50) / 4 = 0.4375
```

This gives the 24-sample batch **equal weight** to the 64-sample batches, which is mathematically incorrect.

#### Correct approach — weighted average

```python
# CORRECT: weight each batch by its size
# Step 1: Convert means back to totals (multiply by n)
tot_loss = (0.45 * 64) + (0.42 * 64) + (0.38 * 64) + (0.50 * 24) = 92.0

# Step 2: Divide by total count
count = 64 + 64 + 64 + 24 = 216
avg_loss = 92.0 / 216 = 0.4259
```

This is exactly what the code does:

```python
tot_loss += loss_func(preds, yb).item() * n   # mean × n = total for this batch
# ...later...
final_loss = tot_loss / count                  # grand total ÷ total samples = correct mean
```

---

## The Full Pattern Visualized

```
For each batch of size n:
    mean_loss = loss_func(preds, yb).item()    # e.g., 0.45
    batch_total = mean_loss * n                 # e.g., 0.45 × 64 = 28.8
    tot_loss += batch_total                     # accumulate totals
    count += n                                  # accumulate sample count

After all batches:
    correct_avg = tot_loss / count              # weighted average across ALL samples
```

This **multiply-by-n then divide-by-total** pattern is the standard way to compute correct dataset-level metrics when batch sizes vary.

In [ ]:
# =====================================================
# TRAIN THE CLASSIFIER
# =====================================================

# Create optimizer
opt = optim.SGD(cnn.parameters(), lr=lr)

# Train for 5 epochs
print("Training CNN classifier on Fashion MNIST...")
print("=" * 50)
loss, acc = fit(5, cnn, F.cross_entropy, opt, dt, dv)

print("=" * 50)
print(f"Final accuracy: {acc:.2%}")

**Fashion MNIST is harder than regular MNIST**, so ~85% accuracy is reasonable for this simple architecture. The important thing is that our data pipeline and training loop work correctly.

---

### Use more CPUs

In [ ]:
def collate_(b):
    return to_device(cf(b))

def data_loaders(dsd, bs, **kwargs):
    return {k: DataLoader(v, bs, num_workers=8, **kwargs) for k, v in dsd.items()}

# But putting things into device as done by collate_ is incompatible with num_workers

The answer to this problem is that we have to rewrite out `fit` function.

In [ ]:
dls = data_loaders(tds, bs, collate_fn=collate_)

In [ ]:
dt = dls['train']
dv = dls['valid']

xb, yb = next(iter(dt))

In [ ]:
labels = ds.features[y].names

In [ ]:
lbl_getter = itemgetter(*yb[:16])
titles = lbl_getter(labels)

---

# Part 4: Building the Autoencoder

---

## Autoencoder Architecture Overview

An autoencoder has two parts:

### 1. Encoder (Compression)
- Takes the input image
- Uses convolutional layers to **reduce spatial size**
- Produces a small "latent" representation

### 2. Decoder (Reconstruction)
- Takes the latent representation
- Uses "deconvolution" layers to **increase spatial size**
- Produces a reconstruction of the original image

```
Input (28×28) → [Encoder] → Latent (small) → [Decoder] → Output (28×28)
                   ↓                              ↓
           Conv layers with              Upsampling layers that
           stride=2 (shrink)             double size (expand)
```

---

## The "Deconvolution" Layer

The decoder needs to **increase** the spatial size. We call this "deconvolution" or "upconvolution", but technically it's:
1. **Upsampling** - Double the size by repeating pixels
2. **Convolution** - Apply a regular conv to smooth/refine

This is more stable than using `nn.ConvTranspose2d` (true deconvolution), which can cause "checkerboard artifacts".

In [ ]:
def deconv(ni, nf, ks=3, act=True):
    """
    Refactored upsampling layer (Deconvolution via Interpolation + Conv).

    Explanation of Spatial Dimensions:
    1. nn.UpsamplingNearest2d(scale_factor=2):
       Doubles the height and width (e.g., 8x8 -> 16x16) by repeating values.

    2. nn.Conv2d(..., stride=1, padding=ks//2):
       With stride=1 and 'same' padding (ks//2), the spatial dimensions
       are PRESERVED. It does NOT reduce the size.
       If input is 16x16, output remains 16x16.
    """
    layers = [
        # Step 1: Double the size (e.g., 8x8 -> 16x16)
        nn.UpsamplingNearest2d(scale_factor=2),

        # Step 2: Convolve to learn features on the larger grid.
        # stride=1: ensures we don't downsample (size stays 16x16).
        # padding=ks//2: offsets the kernel width to keep dimensions identical.
        nn.Conv2d(ni, nf, stride=1, kernel_size=ks, padding=ks//2)
    ]

    if act: layers.append(nn.ReLU())
    return nn.Sequential(*layers)

# Understanding the `deconv` Function

## Overview

The `deconv` function performs **upsampling** — the opposite of what `conv` does. While `conv` shrinks spatial dimensions, `deconv` expands them.

```python
def deconv(ni, nf, ks=3, act=True):
    layers = [
        nn.UpsamplingNearest2d(scale_factor=2),  # Step 1: Double size
        nn.Conv2d(ni, nf, stride=1, kernel_size=ks, padding=ks//2)  # Step 2: Refine
    ]
    if act: layers.append(nn.ReLU())
    return nn.Sequential(*layers)
```

---

## The Two-Step Process

### Step 1: Upsample (Double the Size)

```python
nn.UpsamplingNearest2d(scale_factor=2)
```

This doubles the spatial dimensions by **repeating each pixel**:

```
Input (2×2):          Output (4×4):
┌───┬───┐            ┌───┬───┬───┬───┐
│ A │ B │            │ A │ A │ B │ B │
├───┼───┤     →      ├───┼───┼───┼───┤
│ C │ D │            │ A │ A │ B │ B │
└───┴───┘            ├───┼───┼───┼───┤
                     │ C │ C │ D │ D │
                     ├───┼───┼───┼───┤
                     │ C │ C │ D │ D │
                     └───┴───┴───┴───┘
```

**Problem**: The output looks "blocky" — we just copied pixels. This is where convolution helps.

---

### Step 2: Convolve (Refine the Features)

```python
nn.Conv2d(ni, nf, stride=1, kernel_size=ks, padding=ks//2)
```

With default `ks=3`, this becomes:

```python
nn.Conv2d(ni, nf, stride=1, kernel_size=3, padding=1)
```

This convolution:
1. **Smooths** the blocky upsampled image
2. **Learns** useful features on the larger grid
3. **Preserves** the spatial dimensions (key point!)

---

## Why Does Conv2d Preserve Spatial Dimensions?

### The Parameters

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `stride=1` | 1 | Don't skip any positions |
| `kernel_size=3` | 3×3 | Size of the sliding window |
| `padding=1` | 1 | Add 1 pixel border of zeros |

### The Output Size Formula

$$\text{output} = \left\lfloor \frac{\text{input} + 2 \times \text{padding} - \text{kernel}}{\text{stride}} \right\rfloor + 1$$

For a 16×16 input with our parameters:

$$\text{output} = \left\lfloor \frac{16 + 2(1) - 3}{1} \right\rfloor + 1 = \left\lfloor \frac{15}{1} \right\rfloor + 1 = 16$$

**The output is the same size as the input!**

### Why `padding = ks // 2` Works

This is a general formula for "same" padding:

| Kernel Size | Padding (`ks//2`) | Output Size |
|-------------|-------------------|-------------|
| 3 | 1 | Same as input |
| 5 | 2 | Same as input |
| 7 | 3 | Same as input |

The padding compensates for the kernel "eating into" the edges.

---

## Visual Walkthrough

### Without Padding (padding=0)

```
Input: 16×16
Kernel: 3×3, stride=1, padding=0

The 3×3 kernel can only fit in positions where it doesn't hang off the edge:
  - First valid position: starts at pixel (0,0), covers (0-2, 0-2)
  - Last valid position: starts at pixel (13,13), covers (13-15, 13-15)
  - Total positions: 14 × 14

Output: 14×14 (we lost 2 pixels!)
```

### With Padding (padding=1)

```
Input: 16×16
Kernel: 3×3, stride=1, padding=1

Step 1: Add 1-pixel border of zeros
  - Padded size: 18×18

Step 2: Apply kernel
  - First valid position: starts at pixel (0,0) of padded image
  - Last valid position: starts at pixel (15,15) of padded image
  - Total positions: 16 × 16

Output: 16×16 (same as input!)
```

### Diagram

```
Original 4×4 input:           After padding=1 (now 6×6):
┌───┬───┬───┬───┐            ┌───┬───┬───┬───┬───┬───┐
│ a │ b │ c │ d │            │ 0 │ 0 │ 0 │ 0 │ 0 │ 0 │
├───┼───┼───┼───┤            ├───┼───┼───┼───┼───┼───┤
│ e │ f │ g │ h │            │ 0 │ a │ b │ c │ d │ 0 │
├───┼───┼───┼───┤     →      ├───┼───┼───┼───┼───┼───┤
│ i │ j │ k │ l │            │ 0 │ e │ f │ g │ h │ 0 │
├───┼───┼───┼───┤            ├───┼───┼───┼───┼───┼───┤
│ m │ n │ o │ p │            │ 0 │ i │ j │ k │ l │ 0 │
└───┴───┴───┴───┘            ├───┼───┼───┼───┼───┼───┤
                             │ 0 │ m │ n │ o │ p │ 0 │
                             ├───┼───┼───┼───┼───┼───┤
                             │ 0 │ 0 │ 0 │ 0 │ 0 │ 0 │
                             └───┴───┴───┴───┴───┴───┘

Now 3×3 kernel can start at position (0,0) and still produce 4×4 output!
```

---

## How Channels Change

While spatial dimensions are **preserved**, the number of channels **changes**:

```
Input:  (ni channels, H, W)     e.g., (4, 16, 16)
Output: (nf channels, H, W)     e.g., (2, 16, 16)
```

Each of the `nf` filters produces one output channel:

```
                     Filter 0 ──→ Output Channel 0
Input (4×16×16) ──→  Filter 1 ──→ Output Channel 1
                     
                     Output: (2, 16, 16)
```

---

## Complete `deconv` Transformation

Let's trace through `deconv(4, 2)` with an 8×8 input:

```
Step 1: UpsamplingNearest2d(scale_factor=2)
┌─────────────────┐         ┌─────────────────┐
│   (4, 8, 8)     │   ──→   │   (4, 16, 16)   │
│   4 channels    │         │   4 channels    │
│   8×8 spatial   │         │   16×16 spatial │
└─────────────────┘         └─────────────────┘
        Doubles spatial size, channels unchanged


Step 2: Conv2d(4, 2, stride=1, kernel_size=3, padding=1)
┌─────────────────┐         ┌─────────────────┐
│   (4, 16, 16)   │   ──→   │   (2, 16, 16)   │
│   4 channels    │         │   2 channels    │
│   16×16 spatial │         │   16×16 spatial │
└─────────────────┘         └─────────────────┘
        Spatial preserved, channels: 4 → 2


Step 3: ReLU (if act=True)
┌─────────────────┐         ┌─────────────────┐
│   (2, 16, 16)   │   ──→   │   (2, 16, 16)   │
│   may have      │         │   all values    │
│   negative vals │         │   ≥ 0           │
└─────────────────┘         └─────────────────┘
        Shape unchanged, negative values → 0
```

---

## Comparison: `conv` vs `deconv`

| Aspect | `conv` (Encoder) | `deconv` (Decoder) |
|--------|------------------|-------------------|
| **Spatial change** | Halves (stride=2) | Doubles (upsample) |
| **Channel change** | Usually increases | Usually decreases |
| **Purpose** | Compress | Expand |
| **In autoencoder** | Shrink to bottleneck | Reconstruct from bottleneck |

### In the Autoencoder

```
Encoder (conv):                    Decoder (deconv):
(1, 32, 32)                        (4, 8, 8)
    │ conv(1, 2)                       │ deconv(4, 2)
    ▼                                  ▼
(2, 16, 16)                        (2, 16, 16)
    │ conv(2, 4)                       │ deconv(2, 1)
    ▼                                  ▼
(4, 8, 8)   ─── Bottleneck ───→   (1, 32, 32)
```

---

## Why Not Just Use ConvTranspose2d?

PyTorch has `nn.ConvTranspose2d` which does "true" deconvolution. Why use Upsample + Conv instead?

### The Checkerboard Problem

`ConvTranspose2d` can create **checkerboard artifacts**:

```
Expected output:          ConvTranspose2d output:
┌─────────────────┐       ┌─────────────────┐
│ ░░░░░░░░░░░░░░░ │       │ ░▓░▓░▓░▓░▓░▓░▓░ │
│ ░░░░░░░░░░░░░░░ │       │ ▓░▓░▓░▓░▓░▓░▓░▓ │
│ ░░░░░░░░░░░░░░░ │       │ ░▓░▓░▓░▓░▓░▓░▓░ │
│ ░░░░░░░░░░░░░░░ │       │ ▓░▓░▓░▓░▓░▓░▓░▓ │
└─────────────────┘       └─────────────────┘
   Smooth                    Checkerboard!
```

This happens due to uneven overlap when the kernel "deconvolves."

### Upsample + Conv Avoids This

| Method | Pros | Cons |
|--------|------|------|
| `ConvTranspose2d` | Learnable upsampling | Checkerboard artifacts |
| `Upsample + Conv` | No artifacts, stable | Slightly more computation |

The `Upsample + Conv` approach is now standard in most modern architectures.

---

## Key Takeaways

1. **`stride=1`** — Kernel visits every position, no spatial reduction

2. **`padding=ks//2`** — Compensates for kernel size, preserves dimensions

3. **Two-step process** — Upsample first (double size), then convolve (refine features)

4. **Channel change is independent** — `nf` filters create `nf` output channels regardless of spatial operations

5. **Avoids artifacts** — This approach is more stable than `ConvTranspose2d`

### 🔍 Visualize It: "Same" Padding

Slide the **input size**, **kernel size**, and **padding** to see exactly why `padding = ks//2` makes a stride-1 convolution preserve the spatial size. The zero border and the corner kernel position show *why* it works.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. Loads interactive_viz/padding_playground.html (hosted on
# GitHub Pages) in a full-width iframe that auto-fits its height. How padding changes a conv layer's output size.
# Requires the show_viz() setup cell above.
# ============================================================================
show_viz("interactive_viz/padding_playground.html", height="760px")

### 🔍 Visualize It: Deconv = Upsample + Refine

Step through the two-step deconvolution on a tiny 2×2 patch: first nearest-neighbour **upsampling** (blocky copy), then a stride-1 **conv** that smooths it. The bottom panels compare `ConvTranspose2d` (checkerboard) with the `Upsample + Conv` approach this notebook uses.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. Loads interactive_viz/deconv_upsample_lab.html (hosted on
# GitHub Pages) in a full-width iframe that auto-fits its height. Transposed-conv / upsampling: how the decoder grows feature maps.
# Requires the show_viz() setup cell above.
# ============================================================================
show_viz("interactive_viz/deconv_upsample_lab.html", height="840px")

**Why use Upsampling + Conv instead of ConvTranspose2d?**

| Method | Pros | Cons |
|--------|------|------|
| **ConvTranspose2d** | Learnable upsampling | Can cause checkerboard artifacts |
| **Upsample + Conv** | No artifacts, more stable | Slightly more computation |

The artifacts come from uneven overlap when ConvTranspose2d "deconvolves". Using explicit upsampling avoids this.

---

## Training Functions for Autoencoders

Autoencoders have a different training objective than classifiers:

| Classifier | Autoencoder |
|------------|-------------|
| Input: image | Input: image |
| Target: label (0-9) | Target: **same image** |
| Loss: cross-entropy | Loss: **MSE** (mean squared error) |
| Goal: predict class | Goal: **reconstruct input** |

In [ ]:
# =====================================================
# EVALUATION FUNCTION FOR AUTOENCODERS
# =====================================================

def eval_ae(model, loss_func, valid_dl, epoch=0):
    """
    Evaluate an autoencoder on the validation set.

    Key difference from classifier evaluation:
    - We compare the model's output to the INPUT (xb), not the label
    - The _ in 'for xb, _ in valid_dl' ignores the labels

    Arguments:
        model     - The autoencoder to evaluate
        loss_func - Reconstruction loss function (MSE)
        valid_dl  - Validation data loader
        epoch     - Current epoch number (for display)
    """
    model.eval()  # Set to evaluation mode (disables dropout, etc.)

    with torch.no_grad():  # Don't compute gradients (faster)
        tot_loss, count = 0., 0

        for xb, _ in valid_dl:  # _ ignores labels - we don't use them!
            # Forward pass: try to reconstruct the input
            pred = model(xb)

            n = len(xb)
            count += n

            # Loss: how different is pred from the ORIGINAL INPUT xb?
            # Note: we compare pred to xb, not to labels!
            tot_loss += loss_func(pred, xb).item() * n

    print(f"Epoch {epoch}: loss = {tot_loss/count:.4f}")

# Understanding the Autoencoder Evaluation Function

## Overview

```python
def eval_ae(model, loss_func, valid_dl, epoch=0):
```

This function measures **how well the autoencoder can reconstruct images**. Unlike a classifier that checks "did you guess the right label?", an autoencoder asks "does your output look like the input?"

---

## The Key Insight: Autoencoders vs Classifiers

| Aspect | Classifier | Autoencoder |
|--------|------------|-------------|
| **Input** | Image | Image |
| **Output** | Class label (0-9) | Reconstructed image |
| **Target** | The label `yb` | The input itself `xb` |
| **Question** | "What is this?" | "Can you copy this?" |

```
Classifier:
  Input: 👟 (image of sneaker)
  Output: 7
  Target: 7 (label)
  Loss: Was 7 correct? ✓

Autoencoder:
  Input: 👟 (image of sneaker)
  Output: 👟 (reconstructed sneaker)
  Target: 👟 (the original input!)
  Loss: How similar are they?
```

---

## Line-by-Line Explanation

### 1. Set Evaluation Mode

```python
model.eval()
```

| What It Does | Why It Matters |
|--------------|----------------|
| Switches model to evaluation mode | Disables dropout (if any) |
| | Uses fixed statistics for batch normalization |
| | Gives consistent, reproducible predictions |

**Analogy**: It's like telling a student "this is the real test, not practice anymore."

---

### 2. Disable Gradient Computation

```python
with torch.no_grad():
```

| What It Does | Why It Matters |
|--------------|----------------|
| Stops tracking operations for gradients | Saves memory |
| | Speeds up computation |
| | We don't need gradients — we're not learning |

**Analogy**: You don't need scratch paper when you're just checking answers.

---

### 3. Initialize Counters

```python
tot_loss, count = 0., 0
```

- `tot_loss`: Accumulates the total reconstruction error
- `count`: Counts total number of images evaluated

---

### 4. Loop Through Validation Data

```python
for xb, _ in valid_dl:
```

| Part | Meaning |
|------|---------|
| `xb` | Batch of input images |
| `_` | Labels (we ignore them!) |

**Why ignore labels?** Autoencoders don't care about labels. They just try to reconstruct whatever image you give them.

```
Classifier needs labels:     Autoencoder ignores labels:
  Image: 👟                    Image: 👟
  Label: 7  ← used!            Label: 7  ← ignored!
```

---

### 5. Forward Pass (Reconstruction)

```python
pred = model(xb)
```

The autoencoder tries to reconstruct the input:

```
Input (xb):                    Output (pred):
┌─────────────┐               ┌─────────────┐
│             │               │             │
│   Original  │  ──model──→   │ Reconstruct │
│    Image    │               │    -ion     │
│             │               │             │
└─────────────┘               └─────────────┘
     28×28                         28×28
```

---

### 6. Calculate Reconstruction Loss

```python
tot_loss += loss_func(pred, xb).item() * n
```

**This is the crucial difference from classifiers!**

| Classifier | Autoencoder |
|------------|-------------|
| `loss_func(pred, yb)` | `loss_func(pred, xb)` |
| Compare to **label** | Compare to **input** |

#### What MSE Loss Measures

```python
loss = MSE(pred, xb)  # Mean Squared Error
```

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\text{pred}_i - \text{input}_i)^2$$

For each pixel:
- Calculate difference between predicted and original
- Square it (makes all differences positive)
- Average over all pixels

```
Original pixel: 0.8     Predicted pixel: 0.7
Difference: 0.1
Squared: 0.01

Lower MSE = Better reconstruction
```

---

### 7. Accumulate Statistics

```python
n = len(xb)
count += n
tot_loss += loss_func(pred, xb).item() * n
```

| Variable | Purpose |
|----------|---------|
| `n` | Number of images in this batch |
| `count` | Running total of all images |
| `.item()` | Convert tensor to Python number |
| `* n` | Weight by batch size for proper averaging |

**Why multiply by `n`?**

```
Batch 1: 256 images, loss = 0.05  → contributes 256 × 0.05 = 12.8
Batch 2: 256 images, loss = 0.04  → contributes 256 × 0.04 = 10.24
Batch 3: 100 images, loss = 0.06  → contributes 100 × 0.06 = 6.0
                                     ─────────────────────────────
Total: 612 images                    Total loss: 29.04

Average loss = 29.04 / 612 = 0.0475
```

This handles the case where the last batch might be smaller.

---

### 8. Print Results

```python
print(f"Epoch {epoch}: loss = {tot_loss/count:.4f}")
```

Prints the average reconstruction loss:

```
Epoch 0: loss = 0.0892
Epoch 1: loss = 0.0654
Epoch 2: loss = 0.0521
...
```

**Lower loss = Better reconstruction!**

---

## Visual Summary

```
                    AUTOENCODER EVALUATION
                    
    ┌─────────────────────────────────────────────┐
    │              Validation Data                │
    │  ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐   │
    │  │ 👟  │ │ 👔  │ │ 👗  │ │ 👜  │ │ 🥾  │   │
    │  └─────┘ └─────┘ └─────┘ └─────┘ └─────┘   │
    └────────────────────┬────────────────────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │    AUTOENCODER      │
              │  ┌───────────────┐  │
              │  │   Encoder     │  │
              │  │  (compress)   │  │
              │  └───────┬───────┘  │
              │          │          │
              │  ┌───────▼───────┐  │
              │  │   Bottleneck  │  │
              │  └───────┬───────┘  │
              │          │          │
              │  ┌───────▼───────┐  │
              │  │   Decoder     │  │
              │  │  (reconstruct)│  │
              │  └───────────────┘  │
              └──────────┬──────────┘
                         │
                         ▼
    ┌─────────────────────────────────────────────┐
    │            Reconstructions                  │
    │  ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐   │
    │  │ 👟  │ │ 👔  │ │ 👗  │ │ 👜  │ │ 🥾  │   │
    │  │~blurry~│    │      │      │      │      │
    │  └─────┘ └─────┘ └─────┘ └─────┘ └─────┘   │
    └────────────────────┬────────────────────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │   COMPARE (MSE)     │
              │                     │
              │  Original vs Recon  │
              │  How different?     │
              └──────────┬──────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │  loss = 0.0521      │
              │  (lower = better)   │
              └─────────────────────┘
```

---

## Comparison: Classifier vs Autoencoder Evaluation

```python
# CLASSIFIER evaluation
for xb, yb in valid_dl:
    pred = model(xb)           # Predict class scores
    loss = loss_func(pred, yb) # Compare to LABELS
    acc = accuracy(pred, yb)   # Did we guess right?

# AUTOENCODER evaluation  
for xb, _ in valid_dl:         # Ignore labels!
    pred = model(xb)           # Reconstruct image
    loss = loss_func(pred, xb) # Compare to INPUT
    # No accuracy - just reconstruction quality
```

---

## Key Takeaways

1. **Autoencoders compare output to input** — not to labels

2. **Labels are ignored** — the `_` in `for xb, _ in valid_dl` discards them

3. **MSE measures reconstruction quality** — lower = better

4. **No accuracy metric** — we only measure how close the reconstruction is

5. **Same eval principles apply** — `model.eval()`, `torch.no_grad()`, accumulate statistics

In [ ]:
# =====================================================
# TRAINING FUNCTION FOR AUTOENCODERS
# =====================================================

def fit_ae(epochs, model, loss_func, opt, train_dl, valid_dl):
    """
    Train an autoencoder.

    Key difference from classifier training:
    - Target is the input image itself, not a label
    - We use loss_func(model(xb), xb) instead of loss_func(model(xb), yb)

    Arguments:
        epochs    - Number of training epochs
        model     - The autoencoder to train
        loss_func - Reconstruction loss (MSE)
        opt       - Optimizer
        train_dl  - Training data loader
        valid_dl  - Validation data loader
    """
    for epoch in range(epochs):
        # Training phase
        model.train()

        for xb, _ in train_dl:  # Ignore labels with _
            # Forward pass: reconstruct the input
            pred = model(xb)

            # Loss: compare prediction to ORIGINAL INPUT
            # This is the key autoencoder insight:
            #   We want pred to match xb as closely as possible
            loss = loss_func(pred, xb) # We are trying to recreate the original image

            # Backward pass: compute gradients
            loss.backward()

            # Update weights
            opt.step()
            opt.zero_grad()

        # Evaluation phase
        eval_ae(model, loss_func, valid_dl, epoch)

# Why Sigmoid Output is Correct Here — No Range Mismatch

## Concern

> The model outputs Sigmoid values in `[0, 1]`, but the original images might have a larger range — wouldn't that cause a wrong comparison in `loss_func(model(xb), xb)`?

**Great instinct — but there's no mismatch.** The input images are **already in `[0, 1]` range**.

---

## Where the Scaling Happens

Earlier in the notebook, the data is transformed using `TF.to_tensor`:

```python
def transformi(b):
    b[x] = [TF.to_tensor(o) for o in b[x]]
```

`TF.to_tensor` does three things:

1. Converts the PIL image to a PyTorch tensor
2. Rearranges dimensions from `(H, W, C)` to `(C, H, W)`
3. **Scales pixel values from `[0, 255]` → `[0.0, 1.0]`** by dividing by 255

So by the time data reaches the model, `xb` already lives in `[0, 1]`.

---

## Why Sigmoid Is the Right Choice

The MSE loss computes the **pixel-wise squared difference** between the reconstruction and the original:

```python
loss = F.mse_loss(model(xb), xb)
#                 ↑              ↑
#          Sigmoid output    Original input
#           range [0, 1]     range [0, 1]  ← both match!
```

| Component | Value Range |
|---|---|
| Input `xb` (after `to_tensor`) | `[0.0, 1.0]` |
| Output `model(xb)` (after Sigmoid) | `[0.0, 1.0]` |
| MSE Loss | Compares same-range values ✓ |

If the model output were **unbounded** (no Sigmoid), it could predict values like `-0.3` or `2.5`, which are impossible pixel values. Sigmoid **constrains** the output to the valid range, making the reconstruction physically meaningful.

---

## What Would Go Wrong Without Sigmoid?

Without Sigmoid, the last layer would output raw values from the `deconv` layer, which could be **any real number**. Two problems:

1. **Invalid reconstructions** — pixel values outside `[0, 1]` don't correspond to real images
2. **Harder optimization** — the model would need to learn the correct output range on top of learning the reconstruction, making training slower and less stable

---

## Summary

```
Raw image:    [0, 255]  (integers)
     │
     ▼  TF.to_tensor (÷ 255)
Input xb:     [0.0, 1.0]  (floats)
     │
     ▼  Autoencoder
Output:       [0.0, 1.0]  (floats, thanks to Sigmoid)
     │
     ▼  F.mse_loss(output, input)
Loss:         Both in [0, 1] — fair comparison ✓
```

The Sigmoid isn't causing a mismatch — it's **ensuring** a match. It guarantees the model's output lives in exactly the same range as its input.

# Understanding the Autoencoder Training Function

## Overview

```python
def fit_ae(epochs, model, loss_func, opt, train_dl, valid_dl):
```

This function **teaches the autoencoder to reconstruct images**. The key insight: instead of learning to predict labels, the autoencoder learns to **copy its input through a bottleneck**.

---

## The Core Difference: Classifier vs Autoencoder Training

| Aspect | Classifier | Autoencoder |
|--------|------------|-------------|
| **Goal** | Predict the correct label | Reconstruct the input |
| **Target** | Label `yb` | Input `xb` |
| **Loss** | `loss_func(pred, yb)` | `loss_func(pred, xb)` |
| **Uses labels?** | Yes | No |

```
Classifier Training:
  Input: 👟 → Model → Output: [0.1, 0.0, ..., 0.9, 0.0]
  Target: 7 (sneaker)
  Question: "Did you predict class 7?"

Autoencoder Training:
  Input: 👟 → Model → Output: 👟 (reconstructed)
  Target: 👟 (the same input!)
  Question: "Does your output look like the input?"
```

---

## The Training Loop Structure

```
For each epoch:
    │
    ├── TRAINING PHASE
    │   For each batch:
    │     1. Forward pass (reconstruct)
    │     2. Calculate loss (compare to input)
    │     3. Backward pass (compute gradients)
    │     4. Update weights
    │
    └── EVALUATION PHASE
        Check reconstruction quality on validation data
```

---

## Line-by-Line Explanation

### 1. Loop Through Epochs

```python
for epoch in range(epochs):
```

One epoch = one complete pass through all training data.

```
Epoch 0: See all 60,000 images once
Epoch 1: See all 60,000 images again
Epoch 2: See all 60,000 images again
...
```

---

### 2. Enable Training Mode

```python
model.train()
```

| What It Does | Why It Matters |
|--------------|----------------|
| Activates training behavior | Enables dropout (if present) |
| | Uses batch statistics for normalization |
| | Model knows "I'm learning now" |

---

### 3. Loop Through Training Batches

```python
for xb, _ in train_dl:
```

| Part | Meaning |
|------|---------|
| `xb` | Batch of images (e.g., 256 images) |
| `_` | Labels — **we throw them away!** |

**Why ignore labels?**

The autoencoder doesn't need to know "this is a sneaker" or "this is a dress." It just needs to learn: "whatever comes in, make the same thing come out."

```
Classifier uses both:          Autoencoder uses only images:
  xb (images) ✓                  xb (images) ✓
  yb (labels) ✓                  yb (labels) ✗ (ignored)
```

---

### 4. Forward Pass (Reconstruction)

```python
pred = model(xb)
```

The image goes through the autoencoder:

```
Input Image          Encoder           Bottleneck          Decoder          Output
   (xb)                                                                      (pred)
┌─────────┐       ┌─────────┐       ┌─────────┐       ┌─────────┐       ┌─────────┐
│  28×28  │ ───→  │ Compress│ ───→  │  8×8×4  │ ───→  │ Expand  │ ───→  │  28×28  │
│ Original│       │         │       │ (small!)│       │         │       │  Recon  │
└─────────┘       └─────────┘       └─────────┘       └─────────┘       └─────────┘
  784 values                          256 values                          784 values
```

The bottleneck forces the network to learn **what's important** — it can't memorize everything!

---

### 5. Calculate Loss (The Key Line!)

```python
loss = loss_func(pred, xb)
```

**This is where autoencoders differ from classifiers:**

```python
# Classifier:
loss = loss_func(pred, yb)  # Compare to LABEL

# Autoencoder:
loss = loss_func(pred, xb)  # Compare to INPUT
```

#### What MSE Loss Calculates

```
Original (xb):        Prediction (pred):      Difference:
┌───┬───┬───┐        ┌───┬───┬───┐          ┌───┬───┬───┐
│0.9│0.8│0.7│        │0.8│0.7│0.6│          │0.1│0.1│0.1│
├───┼───┼───┤   vs   ├───┼───┼───┤    →     ├───┼───┼───┤
│0.6│0.5│0.4│        │0.5│0.5│0.4│          │0.1│0.0│0.0│
└───┴───┴───┘        └───┴───┴───┘          └───┴───┴───┘

MSE = average of (differences²) = average of [0.01, 0.01, 0.01, 0.01, 0, 0]
    = 0.0067
```

**Lower MSE = reconstruction looks more like original!**

---

### 6. Backward Pass

```python
loss.backward()
```

PyTorch calculates gradients: **"How should each weight change to reduce the loss?"**

```
loss = 0.0067
       │
       ▼ backward()
┌─────────────────────────────────────┐
│ Gradients computed for every weight │
│                                     │
│ Decoder weights: ∂loss/∂w_decoder   │
│ Encoder weights: ∂loss/∂w_encoder   │
└─────────────────────────────────────┘
```

---

### 7. Update Weights

```python
opt.step()
opt.zero_grad()
```

| Line | What It Does |
|------|--------------|
| `opt.step()` | Adjust all weights using gradients |
| `opt.zero_grad()` | Reset gradients to zero for next batch |

```
Before opt.step():              After opt.step():
weight = 0.5                    weight = 0.5 - lr × gradient
gradient = 0.02                        = 0.5 - 0.1 × 0.02
learning_rate = 0.1                    = 0.498
```

**Why zero gradients?**

Gradients accumulate by default. If we don't reset them, the next batch's gradients would add to the old ones — wrong!

---

### 8. Evaluation Phase

```python
eval_ae(model, loss_func, valid_dl, epoch)
```

After training on all batches, check how well we're doing on data the model hasn't trained on:

```
Training data: Model learns from this
Validation data: Model is tested on this (no learning!)
```

This tells us if we're actually getting better or just memorizing.

---

## Complete Visual Flow

```
                         EPOCH 0
                            │
         ┌──────────────────┴──────────────────┐
         ▼                                     │
┌─────────────────────────────────────┐        │
│         TRAINING PHASE              │        │
│                                     │        │
│  Batch 1: 256 images                │        │
│  ┌─────────────────────────────┐    │        │
│  │ 1. pred = model(xb)         │    │        │
│  │ 2. loss = MSE(pred, xb)     │◄───┼── Compare to INPUT, not label!
│  │ 3. loss.backward()          │    │        │
│  │ 4. opt.step()               │    │        │
│  │ 5. opt.zero_grad()          │    │        │
│  └─────────────────────────────┘    │        │
│                                     │        │
│  Batch 2: 256 images                │        │
│  └─────────────────────────────┘    │        │
│              ...                    │        │
│  Batch N: (remaining images)        │        │
│  └─────────────────────────────┘    │        │
│                                     │        │
└─────────────────┬───────────────────┘        │
                  │                            │
                  ▼                            │
┌─────────────────────────────────────┐        │
│        EVALUATION PHASE             │        │
│                                     │        │
│  eval_ae(model, loss_func, ...)     │        │
│                                     │        │
│  Output: "Epoch 0: loss = 0.0892"   │        │
└─────────────────┬───────────────────┘        │
                  │                            │
                  ▼                            │
              EPOCH 1 ─────────────────────────┘
                  │
                 ...
```

---

## What the Autoencoder Learns

Through this training process, the autoencoder learns:

### Encoder Learns:
- What features are **most important**
- How to **compress** 784 pixels into 256 values
- Which details can be **discarded**

### Decoder Learns:
- How to **reconstruct** from compressed data
- How to **fill in** missing details
- What a "typical" image looks like

```
Training Progress:

Epoch 0:  Input: 👟  →  Output: 🌫️ (blurry blob)     Loss: 0.089
Epoch 5:  Input: 👟  →  Output: 👟 (recognizable)    Loss: 0.045
Epoch 10: Input: 👟  →  Output: 👟 (pretty good!)    Loss: 0.032
```

---

## Side-by-Side: Classifier vs Autoencoder Training

```python
# ==================== CLASSIFIER ====================
def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:        # ← Uses labels!
            pred = model(xb)
            loss = loss_func(pred, yb)  # ← Compare to LABEL
            loss.backward()
            opt.step()
            opt.zero_grad()
        # evaluate with accuracy...

# ==================== AUTOENCODER ====================
def fit_ae(epochs, model, loss_func, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        model.train()
        for xb, _ in train_dl:         # ← Ignores labels!
            pred = model(xb)
            loss = loss_func(pred, xb)  # ← Compare to INPUT
            loss.backward()
            opt.step()
            opt.zero_grad()
        eval_ae(model, loss_func, valid_dl, epoch)
```

**The only differences:**
1. `yb` → `_` (ignore labels)
2. `loss_func(pred, yb)` → `loss_func(pred, xb)` (compare to input)

---

## Key Takeaways

1. **Labels are ignored** — Autoencoders are **unsupervised** (no labels needed)

2. **Target is the input** — `loss_func(pred, xb)` compares output to input

3. **Bottleneck forces learning** — The network must figure out what's important to compress

4. **Same training mechanics** — Forward, backward, update still work the same way

5. **Lower loss = better reconstruction** — The goal is to make output match input

---

## Building the Autoencoder Architecture

Now let's design the actual autoencoder. We need to handle the 28×28 input carefully:

**Problem:** 28 doesn't divide evenly by 2 multiple times.
- 28 → 14 → 7 → 3.5 (not integer!)

**Solution:** Pad to 32×32 first, then downsample/upsample, then crop back.
- 32 → 16 → 8 → 16 → 32 (all integers!)

In [ ]:
# =====================================================
# AUTOENCODER ARCHITECTURE
# =====================================================

ae = nn.Sequential(
    # ===== PREPROCESSING =====
    # Pad 28x28 to 32x32 by adding 2 pixels on each side
    # ZeroPad2d(2) adds 2 zeros to left, right, top, bottom
    nn.ZeroPad2d(2),        # 28x28 -> 32x32

    # ===== ENCODER (Compress) =====
    # conv(1, 2): 1 input channel -> 2 output channels, stride=2
    conv(1, 2),              # 32x32 -> 16x16, 2 channels
    conv(2, 4),              # 16x16 -> 8x8,   4 channels
    # At this point, we have an 8x8x4 "latent representation"
    # That's 256 values (compared to 784 original pixels)

    # ===== DECODER (Reconstruct) =====
    # deconv layers double the spatial size
    deconv(4, 2),            # 8x8   -> 16x16, 2 channels
    deconv(2, 1, act=False), # 16x16 -> 32x32, 1 channel (no ReLU!)

    # ===== POSTPROCESSING =====
    # Crop back from 32x32 to 28x28
    # ZeroPad2d(-2) removes 2 pixels from each side
    nn.ZeroPad2d(-2),        # 32x32 -> 28x28

    # Sigmoid squashes output to [0, 1] range (like our input)
    nn.Sigmoid()
).to(def_device)

print("Autoencoder Architecture:")
print("=" * 50)
print(ae)

### Understanding the Architecture

Let's trace through the shapes:

| Layer | Operation | Shape | Notes |
|-------|-----------|-------|-------|
| Input | - | (1, 28, 28) | Original image |
| ZeroPad2d(2) | Pad | (1, 32, 32) | Add border |
| conv(1, 2) | Encode | (2, 16, 16) | Halve size |
| conv(2, 4) | Encode | (4, 8, 8) | **Bottleneck!** |
| deconv(4, 2) | Decode | (2, 16, 16) | Double size |
| deconv(2, 1) | Decode | (1, 32, 32) | Double size |
| ZeroPad2d(-2) | Crop | (1, 28, 28) | Remove border |
| Sigmoid | Activate | (1, 28, 28) | Squash to [0,1] |

**The bottleneck** at 8×8×4 = 256 values is where the compression happens. The original image has 28×28 = 784 pixels, so we're compressing to about 1/3 the size!

### 🔍 Visualize It: The Architecture in 3D

Each layer is a 3-D block — width × height are the spatial size, depth is the channel count. **Drag to rotate, scroll to zoom.** Step through the layers (or click one on the rail) to watch the volumes shrink to the bottleneck and grow back out into the classic hourglass.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. Loads interactive_viz/ae_3d_bottleneck.html (hosted on
# GitHub Pages) in a full-width iframe that auto-fits its height. 3-D view of the bottleneck (latent) representation.
# Requires the show_viz() setup cell above.
# ============================================================================
show_viz("interactive_viz/ae_3d_bottleneck.html", height="800px")

In [ ]:
# =====================================================
# VERIFY THE ARCHITECTURE
# =====================================================

# Test with a batch of images
test_output = ae(xb)

print(f"Input shape:  {xb.shape}")
print(f"Output shape: {test_output.shape}")
print(f"\nShapes match: {xb.shape == test_output.shape}")
print(f"\nOutput range: [{test_output.min():.3f}, {test_output.max():.3f}]")
print("(Should be in [0, 1] due to Sigmoid)")

---

# Part 5: Training the Autoencoder

---

## The Loss Function: MSE

We use **Mean Squared Error (MSE)** as our loss function:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\text{pred}_i - \text{target}_i)^2$$

This measures the average squared difference between each predicted pixel and the corresponding original pixel. Lower MSE = better reconstruction.

In [ ]:
# =====================================================
# EVALUATE BEFORE TRAINING (RANDOM WEIGHTS)
# =====================================================

print("Loss before training (random weights):")
eval_ae(ae, F.mse_loss, dv)

The initial loss is high because the network outputs random noise. Let's train it!

In [ ]:
# =====================================================
# TRAINING PHASE 1: LOW LEARNING RATE
# =====================================================

# Start with a low learning rate to be safe
opt = optim.SGD(ae.parameters(), lr=0.01)

print("Phase 1: Training with lr=0.01")
print("=" * 40)
fit_ae(5, ae, F.mse_loss, opt, dt, dv)

In [ ]:
# =====================================================
# TRAINING PHASE 2: HIGHER LEARNING RATE
# =====================================================

# Now that the network is partially trained, we can use a higher LR
opt = optim.SGD(ae.parameters(), lr=0.1) # opt = optim.AdamW(ae.parameters(), lr=0.1)

print("\nPhase 2: Training with lr=0.1")
print("=" * 40)
fit_ae(5, ae, F.mse_loss, opt, dt, dv)

**The loss decreased significantly!** Let's see what the reconstructions look like.

---

# Part 6: Visualizing the Results

---

In [ ]:
# =====================================================
# GENERATE RECONSTRUCTIONS
# =====================================================

# Run a batch through the autoencoder
ae.eval()  # Set to evaluation mode
with torch.no_grad():
    reconstructions = ae(xb)

print(f"Original images shape: {xb.shape}")
print(f"Reconstructions shape: {reconstructions.shape}")

In [ ]:
# =====================================================
# DISPLAY RECONSTRUCTIONS
# =====================================================

print("Reconstructed Images:")
show_images(reconstructions[:16].cpu(), nrows=2, ncols=8, imsize=1.5)

In [ ]:
# =====================================================
# DISPLAY ORIGINAL IMAGES FOR COMPARISON
# =====================================================

print("Original Images:")
show_images(xb[:16].cpu(), nrows=2, ncols=8, imsize=1.5)

In [ ]:
# =====================================================
# SIDE-BY-SIDE COMPARISON
# =====================================================

# Let's show original and reconstruction side by side
fig, axes = plt.subplots(4, 8, figsize=(16, 8))

for i in range(8):
    # Top row: original
    show_image(xb[i].cpu(), ax=axes[0, i])
    if i == 0:
        axes[0, i].set_ylabel("Original", fontsize=12)

    # Second row: reconstruction
    show_image(reconstructions[i].cpu(), ax=axes[1, i])
    if i == 0:
        axes[1, i].set_ylabel("Reconstructed", fontsize=12)

    # Third row: another set of originals
    show_image(xb[i+8].cpu(), ax=axes[2, i])
    if i == 0:
        axes[2, i].set_ylabel("Original", fontsize=12)

    # Fourth row: their reconstructions
    show_image(reconstructions[i+8].cpu(), ax=axes[3, i])
    if i == 0:
        axes[3, i].set_ylabel("Reconstructed", fontsize=12)

plt.suptitle("Autoencoder: Original vs Reconstructed", fontsize=14)
plt.tight_layout()

**Observations:**

1. The reconstructions capture the **overall shape** of each clothing item
2. **Details are smoothed out** - this is expected with compression
3. The network learned to preserve the most important features
4. Some items reconstruct better than others (simpler shapes = easier)

---

In [ ]:
# =====================================================
# COMPUTE PER-IMAGE RECONSTRUCTION ERROR
# =====================================================

# Calculate MSE for each image individually
with torch.no_grad():
    # Compute squared differences
    sq_diff = (reconstructions - xb) ** 2
    # Average over each image (dims 1, 2, 3 are channel, height, width)
    per_image_mse = sq_diff.mean(dim=(1, 2, 3))

print("Reconstruction error (MSE) for first 16 images:")
for i in range(16):
    label = labels[yb[i].item()]
    print(f"  Image {i:2d} ({label:12s}): MSE = {per_image_mse[i]:.4f}")

---

# Part 7: Understanding What the Autoencoder Learned

---

## The Latent Space

The middle of the autoencoder (after the encoder, before the decoder) contains a compressed representation. Let's examine it.

In [ ]:
# =====================================================
# EXAMINING THE LATENT REPRESENTATION
# =====================================================

# Create just the encoder part
encoder = nn.Sequential(
    ae[0],  # ZeroPad2d
    ae[1],  # conv(1, 2)
    ae[2],  # conv(2, 4)
).to(def_device)

# Get the latent representation of our batch
with torch.no_grad():
    latent = encoder(xb)

print(f"Input shape:  {xb.shape}")
print(f"Latent shape: {latent.shape}")
print(f"\nCompression ratio: {xb.numel() / latent.numel():.1f}x")
print(f"  Original:   {28*28} = 784 values per image")
print(f"  Compressed: {8*8*4} = 256 values per image")

In [ ]:
# =====================================================
# VISUALIZE THE LATENT CHANNELS
# =====================================================

# Look at the 4 latent channels for one image
img_idx = 0

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

# Original image
show_image(xb[img_idx].cpu(), ax=axes[0])
axes[0].set_title(f"Original\n{labels[yb[img_idx].item()]}")

# 4 latent channels
for c in range(4):
    show_image(latent[img_idx, c].cpu(), ax=axes[c+1], noframe=False)
    axes[c+1].set_title(f"Latent Channel {c}")

plt.suptitle("Latent Representation (8x8 per channel)", fontsize=14)
plt.tight_layout()

**Each latent channel captures different aspects of the image!** The decoder learns to combine these channels to reconstruct the original.

---

### 🔍 Visualize It: Walking Through the Latent Space

This is the payoff. Drag the marker around the compressed latent space (or click a cluster) and watch the decoder rebuild an image from that exact point. Moving *between* clusters morphs one garment into another — the sign of a well-organized latent space.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. Loads interactive_viz/latent_space_explorer.html (hosted on
# GitHub Pages) in a full-width iframe that auto-fits its height. Walk the latent space and watch the decoded output change.
# Requires the show_viz() setup cell above.
# ============================================================================
show_viz("interactive_viz/latent_space_explorer.html", height="760px")

---

# Summary

---

## Key Concepts Learned

### What is an Autoencoder?
- A neural network that learns to **compress** and **reconstruct** data
- Consists of an **encoder** (compress) and **decoder** (expand)
- The **bottleneck** forces learning of essential features

### Architecture Components

| Component | Purpose | Implementation |
|-----------|---------|----------------|
| **Encoder** | Compress input | Conv layers with stride=2 |
| **Bottleneck** | Compressed representation | Smallest layer in the middle |
| **Decoder** | Reconstruct output | Upsample + Conv layers |
| **Sigmoid** | Bound output to [0,1] | Final activation |

### Deconvolution (Upsampling)
- Opposite of convolution with stride
- We use: **Upsample + Conv** (stable, no artifacts)
- Alternative: `ConvTranspose2d` (can cause checkerboard artifacts)

### Training Autoencoders
- **Loss function**: MSE (Mean Squared Error)
- **Target**: The input image itself!
- **Goal**: Minimize difference between input and reconstruction

### Applications
- **Dimensionality reduction** - Like PCA but non-linear
- **Denoising** - Train with noisy input, clean target
- **Anomaly detection** - High reconstruction error = unusual
- **Feature learning** - Encoder learns useful representations
- **Generative models** - VAEs extend this to generate new data

---

## What's Next?

In future notebooks, we'll explore:
- **Variational Autoencoders (VAEs)** - Generate new images
- **Deeper architectures** - Better reconstructions
- **Different loss functions** - Perceptual loss, adversarial loss
- **Applications** - Denoising, super-resolution, style transfer

---